# Notebook 07 — Closing Output Generator

Seed a tiny Claim with one Claimed Condition, persist a Lay Statement + Exam Prep, then call `generate_session_output` and print the resulting Markdown bundle.

**Prerequisite:** Neo4j up (`make up`).


In [ ]:
from pathlib import Path
import tempfile

from va_agent.graph.driver import GraphDriver
from va_agent.graph.tools import record_measurement, record_symptom, record_veteran, reset_user
from va_agent.output.closing import generate_session_output
from va_agent.output.exam_prep import generate_exam_prep, persist_exam_prep, persist_lay_statement
from va_agent.review.claim_reviewer import create_claim

USER = 'notebook-07-demo-user'
DC = 'TEST5260NB'

driver = GraphDriver.from_env()

## 1. Seed the graph

A minimal CFR slice: one DC with one 10% rating level whose Criterion has a flexion measurement. Plus a veteran with a matching SymptomReport + MeasurementReport.


In [ ]:
driver.cfr_write('MATCH (dc:CFR:DiagnosticCode {code: $code}) DETACH DELETE dc', params={'code': DC})
driver.cfr_write(
    '''
    MERGE (s:CFR:Section {id: 'NB.4.71'})
    MERGE (dc:CFR:DiagnosticCode {code: $code})
      SET dc.title = 'Leg, limitation of flexion (demo)', dc.body_system = 'musculoskeletal'
    MERGE (dc)-[:IN_SECTION]->(s)
    MERGE (rl:CFR:RatingLevel {percent: 10})
    MERGE (dc)-[:HAS_RATING]->(rl)
    MERGE (c:CFR:Criterion {text: 'Flexion limited to 45 degrees'})
    MERGE (rl)-[:REQUIRES]->(c)
    MERGE (c)-[:CRITERION_FOR]->(dc)
    MERGE (a:CFR:Anatomy {name: 'demo-knee'}) SET a.body_system = 'musculoskeletal'
    MERGE (m:CFR:Measurement {
        name: 'flexion', body_part: 'demo-knee',
        operator: '<=', value: 45.0, unit: 'degrees'
    })
    MERGE (c)-[:HAS_MEASUREMENT]->(m)
    MERGE (m)-[:OF_ANATOMY]->(a)
    ''',
    params={'code': DC},
)

reset_user(driver, USER)
record_veteran(driver, USER, branch='Army', deployments=['Iraq 2008'], discharge_characterization='Honorable')
record_symptom(
    driver, USER,
    text='right knee locks up when crouching', body_part='demo-knee',
    typical_severity='moderate', flareup_severity='severe',
    flareup_frequency='weekly', flareup_duration='2 days',
    functional_loss=['cannot kneel', 'cannot squat'],
)
record_measurement(driver, USER, name='flexion', body_part='demo-knee', value=40, unit='degrees')

claim_id = create_claim(driver, USER, [DC])
persist_lay_statement(
    driver, USER, claim_id=claim_id, dc_code=DC,
    text='My right knee flexion is limited to 40 degrees on bad days, with severe weekly flare-ups lasting two days. I cannot kneel or squat.',
    factuality_ok=True,
)
prep = generate_exam_prep(driver, USER, DC)
persist_exam_prep(driver, prep, claim_id=claim_id)
print('Claim seeded:', claim_id)

## 2. Generate the Closing Output


In [ ]:
out_dir = Path(tempfile.mkdtemp(prefix='va-closing-'))
session_dir = generate_session_output(driver, USER, claim_id, out_dir)
print('Generated under:', session_dir)
for p in sorted(session_dir.iterdir()):
    print(' ', p.name)

## 3. Show the per-condition file


In [ ]:
from IPython.display import Markdown
Markdown((session_dir / f'dc-{DC}.md').read_text())

## 4. README + filing steps


In [ ]:
Markdown((session_dir / 'README.md').read_text())

In [ ]:
Markdown((session_dir / 'va-gov-filing-steps.md').read_text())

## 5. Cleanup


In [ ]:
reset_user(driver, USER)
driver.cfr_write('MATCH (dc:CFR:DiagnosticCode {code: $code}) DETACH DELETE dc', params={'code': DC})
driver.cfr_write("MATCH (a:CFR:Anatomy {name: 'demo-knee'}) DETACH DELETE a")
driver.cfr_write("MATCH (s:CFR:Section {id: 'NB.4.71'}) DETACH DELETE s")
driver.close()
print('Cleaned up.')